# Auto tune

The bench day: commission, identify, tune, write, verify.

In [1]:
SIMULATED = True          # False, and PORT, at the bench
PORT = 'COM4'

In [2]:
from coaxial import Coaxial63100

device = Coaxial63100(port=PORT, simulated_device=SIMULATED).open()
print(device)

<Coaxial63100 Simulated SIMULATED>


`coaxial.commission.Commissioning` is the procedure `tools/commission.py` runs: the AFE noise floor, the sample point, the offsets, the gain mismatch, the sign, the dead time, the inductance map, the flux, the injection budget, the gains, the decision, the verification. `arm` is what `gates.arm()` is called with when a step needs the stage.

In [3]:
from coaxial.commission import Commissioning

c = Commissioning(device, arm=dict(bypass_sto=True, ignore_interlock=True),
                  log=print, rated_rpm=3000.0)
report = c.run()
print(report['line'])

gate supply {'volts': 11.055, 'powered': True}


zero-speed: yes (SNR 20 dB), min closed-loop 3 %, iloop 2500 Hz, sigma_theta 11.2 deg at 0 %


In [4]:
r = report['results']
print('steps that ran:', ', '.join(sorted(r)))
print()
print('sigma_i      %.4f A, ENOB %.1f' % (r['afe']['sigma_i'], r['afe']['enob']))
print('sample point CCR5 %d of %d' % (r['sample_point']['best'], r['sample_point']['period']))
print('decision     %s' % r['decision']['method'])
print('verify       %s' % {k: r['verify'][k] for k in
                           ('method', 'sigma_theta_deg', 'omega_hat', 'fault')})

steps that ran: afe, budget, deadtime, decision, flux, gains, l_map, offsets, polarity, sample_point, sign, verify

sigma_i      0.1616 A, ENOB 9.5
sample point CCR5 2360 of 2376
decision     injection
verify       {'method': 'injection', 'sigma_theta_deg': 11.210459914286835, 'omega_hat': 0.0, 'fault': None}


Identified, as a `Parameters`: what the steps measured where they measured it, and the record's own value where a step reported no current. `measured` and `source` travel with it.

In [5]:
from coaxial.motor import Parameters

p = device.drive.params()

def got(step, key, fallback):
    block = r.get(step) or {}
    return block[key] if block.get('measured') is not False and key in block else fallback

identified = Parameters(
    name='commissioned',
    r=got('deadtime', 'r', p['motor_r_uohm']),
    ld=got('l_map', 'ld', p['motor_ld_nh']),
    lq=got('l_map', 'lq', p['motor_lq_nh']),
    lam=got('flux', 'lambda', p['motor_lambda_uvs']),
    poles=int(p['motor_pole_pairs']),
    sat=0.3, i_sat=4.0, measured=True, source='commissioning on this rig')
print(identified)
vdc = device.drive.state()['vdc']
print('link %.2f V' % vdc)

<commissioned measured: R 0.0514 ohm, Ld 19.5 uH, Lq 29.4 uH, lambda 0.00546 Wb, 7 pole pairs, KV 144>
link 24.77 V


A robust tune searched for exactly that machine: `montecarlo.run_job` takes the motor and the limits in the job, so the search sizes itself to it. Small here; the tool's defaults are larger.

In [6]:
import os
import sys

sys.path.insert(0, os.path.join('..', 'host', 'tools'))
import montecarlo as mc

fields = {k: getattr(identified, k) for k in ('name', 'r', 'ld', 'lq', 'lam', 'poles',
                                             'j', 'b', 'sat', 'i_sat', 'measured', 'source')}
i_max, i_trip = p['drv_i_max_ma'], p['drv_i_trip_ma']
knobs = mc.candidates(6, seed=3)
jobs = [{'vdc': vdc, 'knobs': k, 'seed': 1000 * i + s, 'motor': fields,
         'i_max': i_max, 'i_trip': i_trip, 'i_h_max': 1.0, 'k_prop': 0.0}
        for i, k in enumerate(knobs) for s in range(3)]
with mc.pool() as pool:
    runs = mc.sweep(pool, jobs)
score = mc.score(runs)
best = score.loc[score.robust.idxmin()]
print(score[['robust', 'mean', 'p90'] + list(mc.KNOBS)].round(4).sort_values('robust').head())

      1 / 18 runs, 1 s
      2 / 18 runs, 1 s
      3 / 18 runs, 1 s
      4 / 18 runs, 1 s


      5 / 18 runs, 1 s
      6 / 18 runs, 1 s
      7 / 18 runs, 1 s
      8 / 18 runs, 1 s
      9 / 18 runs, 1 s
     10 / 18 runs, 1 s
     11 / 18 runs, 1 s
     12 / 18 runs, 1 s
     13 / 18 runs, 1 s
     14 / 18 runs, 1 s
     15 / 18 runs, 1 s
     16 / 18 runs, 1 s
     17 / 18 runs, 1 s
     18 / 18 runs, 1 s
    robust     mean      p90       bw_i     f_pll    zeta   v_inj  n_inj  \
4   0.6579   0.2594   0.3985  1538.2594   71.1231  1.0870  0.1301      4   
0   2.5824   1.0718   1.5106   326.3185   47.1383  0.8218  0.0399      2   
5   4.3618   1.7696   2.5923  1796.9124   22.0094  0.6241  0.0511      1   
2   4.9502   2.4033   2.5469   693.1668  117.6864  1.0018  0.1415      2   
1  27.3641  13.2225  14.1416   517.7425  179.6790  0.5783  0.0233      2   

       w_lo  w_ratio     bw_w  
4  171.8142   3.1210   7.8188  
0  472.5363   2.4998   1.5650  
5  215.6562   2.0760   1.6642  
2   74.2742   2.1729   3.3597  
1  125.4686   3.7706  16.4201  


Written to the record in the record's own names, and the drive verifies itself under the tune.

In [7]:
tune = mc.design({k: best[k] for k in mc.KNOBS}, vdc, identified, i_max, i_trip, 1.0)
written = device.drive.set_params(
    motor_r_uohm=identified.r, motor_ld_nh=identified.ld, motor_lq_nh=identified.lq,
    motor_lambda_uvs=identified.lam,
    drv_kp_mv_per_a=tune['kp'], drv_ki_v_per_as=tune['ki'],
    drv_l1_milli=tune['l1'], drv_l2_milli=tune['l2'],
    drv_inj_mv=tune['inj_volts'], drv_inj_periods=tune['inj_periods'],
    drv_eps_gain_ua_per_rad=tune['eps_gain'],
    drv_w_lo_mrad_s=tune['w_lo'], drv_w_hi_mrad_s=tune['w_hi'])
for name, value in written.items():
    print('%-24s %s' % (name, value))
check = c.verify(iq=0.5, seconds=1.0)
print({k: check[k] for k in ('method', 'sigma_theta_deg', 'ljung_box', 'omega_hat', 'fault')})

motor_r_uohm             0.05141539593734577
motor_ld_nh              1.9545918367346942e-05
motor_lq_nh              2.9443877551020412e-05
motor_lambda_uvs         0.005462377155532926
drv_kp_mv_per_a          0.18891460655628345
drv_ki_v_per_as          496.93849692249677
drv_l1_milli             0.15543560129637823
drv_l2_milli             31.952201905380214
drv_inj_mv               0.4886479591836735
drv_inj_periods          4.0
drv_eps_gain_ua_per_rad  0.16808178825160286
drv_w_lo_mrad_s          171.8141704274725
drv_w_hi_mrad_s          536.2273764698308


{'method': 'injection', 'sigma_theta_deg': 1.7044017709793713, 'ljung_box': {'q': 2.0679059625118126, 'threshold': 14.067, 'lags': 7, 'white': True}, 'omega_hat': 0.0, 'fault': None}


`device.calibration.save()` is what keeps the record across a reset - the drive reloads its parameters from it at boot, so a board runs the same tune after a power cycle that it ran before.

In [8]:
print('saved:', device.calibration.save())
print('the drive reads them back through the record:')
for name, value in sorted(device.drive.params().items()):
    print('   %-26s %s' % (name, value))
device.close()

saved: True
the drive reads them back through the record:
   drv_dt_step_ma             0.5714285714285714
   drv_eps_gain_ua_per_rad    0.16808178825160286
   drv_i_max_ma               5.0
   drv_i_trip_ma              100.0
   drv_inj_mv                 0.4886479591836735
   drv_inj_periods            4.0
   drv_inj_phase_mrad         0.0
   drv_ki_v_per_as            496.93849692249677
   drv_kp_mv_per_a            0.18891460655628345
   drv_l1_milli               0.15543560129637823
   drv_l2_milli               31.952201905380214
   drv_sigma_i_ua             0.1616215257549312
   drv_sign                   1.0
   drv_trigger_ticks          2360.0
   drv_v_frac_ppm             0.95
   drv_w_hi_mrad_s            536.2273764698308
   drv_w_lo_mrad_s            171.8141704274725
   motor_lambda_uvs           0.005462377155532926
   motor_ld_nh                1.9545918367346942e-05
   motor_lq_nh                2.9443877551020412e-05
   motor_pole_pairs           7.0
   motor_r_uohm 

## Conclusions

In [9]:
def step(number, name, got, line):
    if isinstance(got, dict) and got.get('measured') is False:
        print('%-2s %-13s not measured - %s' % (number, name, got.get('why', 'no current')))
    else:
        print('%-2s %-13s %s' % (number, name, line()))

step(1, 'AFE', r['afe'], lambda: 'sigma_i %.4f A, ENOB %.1f, ISR %.1f us'
     % (r['afe']['sigma_i'], r['afe']['enob'], r['afe']['latency']['isr_cost_us']))
step(2, 'sample point', r['sample_point'], lambda: 'CCR5 %d of %d (was %d)'
     % (r['sample_point']['best'], r['sample_point']['period'], r['sample_point']['was']))
step(3, 'offsets', r['offsets'], lambda: ' '.join(
    '%s %+d%s' % (n[-1], v['offset_raw'], ' SUSPECT' if v['suspect'] else '')
    for n, v in r['offsets'].items()))
step(4, 'sign', r['sign'], lambda: 'sign %+d, id %.3f A'
     % (r['sign']['sign'], r['sign']['id']))
step(5, 'dead time', r['deadtime'], lambda: 'R %.4f ohm, V_dt %.3f V, knee %.2f A'
     % (r['deadtime']['r'], r['deadtime']['v_dt'], r['deadtime']['i_knee']))
step(6, 'L map', r['l_map'], lambda: 'Ld %.1f uH, Lq %.1f uH, dL/L %.3f'
     % (r['l_map']['ld'] * 1e6, r['l_map']['lq'] * 1e6, r['l_map']['dl_over_l']))
step(7, 'flux', r['flux'], lambda: 'lambda %.5f V.s, load angle %.2f rad'
     % (r['flux']['lambda'], r['flux']['load_angle']))
step(8, 'budget', r['budget'], lambda: 'f_inj %.0f Hz, V %.2f, SNR %.1f dB, limited by %s'
     % (r['budget']['choice']['f_inj_hz'], r['budget']['choice']['v_inj'],
        r['budget']['choice']['snr_db'], r['budget']['choice']['limited_by']))
step(9, 'gains', r['gains'], lambda: 'iloop %.0f Hz, PLL %.0f Hz, crossover %.0f rpm'
     % (r['gains']['loop']['bw_hz'], (r['gains']['kalman'] or {}).get('wn_hz', 0.0),
        r['gains']['crossover']['rpm']))
step(10, 'decision', r['decision'], lambda: '%s (SNR %.1f dB against %.0f)'
     % (r['decision']['method'], r['decision']['snr_db'], r['decision']['threshold_db']))
step(11, 'verify', check, lambda: 'sigma_theta %.2f deg, innovation %s, fault %s'
     % (check['sigma_theta_deg'],
        'white' if check['ljung_box']['white'] else 'NOT white', check['fault']))
print()
print('tune written for %.1f V:' % vdc)
for name, value in sorted(written.items()):
    print('   %-26s %s' % (name, value))

1  AFE           sigma_i 0.1616 A, ENOB 9.5, ISR 3.4 us
2  sample point  CCR5 2360 of 2376 (was 2360)
3  offsets       U +1400 V -8030 SUSPECT W +360
4  sign          sign +1, id 6.000 A
5  dead time     R 0.0514 ohm, V_dt 0.497 V, knee 0.30 A
6  L map         Ld 19.5 uH, Lq 29.4 uH, dL/L 0.202
7  flux          lambda 0.00546 V.s, load angle -0.42 rad
8  budget        f_inj 25000 Hz, V 0.30, SNR 20.0 dB, limited by target
9  gains         iloop 2500 Hz, PLL 71 Hz, crossover 94 rpm
10 decision      injection (SNR 20.0 dB against 10)
11 verify        sigma_theta 1.70 deg, innovation white, fault None

tune written for 24.8 V:
   drv_eps_gain_ua_per_rad    0.16808178825160286
   drv_inj_mv                 0.4886479591836735
   drv_inj_periods            4.0
   drv_ki_v_per_as            496.93849692249677
   drv_kp_mv_per_a            0.18891460655628345
   drv_l1_milli               0.15543560129637823
   drv_l2_milli               31.952201905380214
   drv_w_hi_mrad_s            536.227

The order is not a preference: each step needs what the one before it measured. The sample point cannot be scanned without a stage - with nothing switching the scan is a walk through noise and picked 990 of 2376 off exactly that. The offsets need the sample point; the gain mismatch needs the offsets; the injection budget needs sigma_i and the inductances; the Kalman gains need the budget's demodulator gain and the measured noise, so the observer's bandwidth comes out of the noise rather than a knob.

An offset past `limit_codes` is reported and **not** applied: a phase reading -52 A with nothing connected is a fault, and zeroing it would hide it. `decide` picks injection only when the budget clears the threshold; below it the start is I/f with a saturation pulse for the polarity, which is what only saturation can settle - injection locks the d **axis**, not which end of it is the magnet.

The verification judges the innovation rather than the answer: Ljung-Box on the autocorrelation, and the deviation as the sigma_theta proxy. `run()` puts the stage down whatever happens.

The search then sizes itself to what was just identified - `run_job` takes the motor and the board's limits in the job - and what it finds goes into the record in the record's own names, which is where the drive reads it (invariant 7).